In [ ]:
# --- dataset download from the GitHub Release of the project
#     NOT from Google Drive: the public Drive endpoint applies a per-file download quota on the
#     bytes served and blocks Colab datacenter IPs with "Cannot retrieve the public link of the
#     file". A GitHub Release asset has no quota, needs no login and no mounted Drive.
#
#     Doppler dataset: SHARP (Meneghello et al.), mirrored in the release for reproducibility only.
import os, glob, shutil, subprocess

RELEASE = ('https://github.com/marchettialessio/Human-Activity-Recognition-with-Wi-Fi'
           '/releases/download/assets-v1')

DATASET_ZIP = 'doppler_traces.zip'
DATASET_SIZE = 799057851          # exact size in bytes: guards against a truncated download


def ok_size(path, size):
    return os.path.isfile(path) and os.path.getsize(path) == size


def _resume_cmd(url, dst):
    """wget on Colab, curl elsewhere: both resume a partial file instead of restarting it."""
    if shutil.which('wget'):
        return ['wget', '-q', '--show-progress', '-c', '-O', dst, url]
    if shutil.which('curl'):
        return ['curl', '-fL', '-#', '-C', '-', '-o', dst, url]
    raise RuntimeError('neither wget nor curl available')


def fetch(name, size, out_dir='/content', tries=3):
    """Download a Release asset and check its exact size: a truncated file would break torch.load
       or, worse, make the training silently restart from epoch 0."""
    dst = os.path.join(out_dir, name)
    for k in range(tries):
        if ok_size(dst, size):
            return dst
        if os.path.isfile(dst) and os.path.getsize(dst) > size:
            os.remove(dst)        # corrupt (too big): resuming would never fix it
        r = subprocess.run(_resume_cmd(f'{RELEASE}/{name}', dst))
        got = os.path.getsize(dst) if os.path.isfile(dst) else 0
        if ok_size(dst, size):
            return dst
        print(f'  {name}: attempt {k + 1}/{tries} incomplete '
              f'({got}/{size} bytes, exit={r.returncode})')
    raise RuntimeError(f'{name}: download failed, re-run this cell (the partial file is resumed)')


if not os.path.isdir('/content/doppler_traces'):
    z = fetch(DATASET_ZIP, DATASET_SIZE)
    !unzip -q -o "{z}" -d /content
    # the zip may unpack into a subfolder: normalize to /content/doppler_traces
    if not os.path.isdir('/content/doppler_traces'):
        cand = glob.glob('/content/**/doppler_traces', recursive=True)
        assert cand, 'doppler_traces not found inside the zip'
        !mv "{cand[0]}" /content/doppler_traces

assert os.path.isdir('/content/doppler_traces'), 'dataset not found after download'
print('Dataset:', os.path.isdir('/content/doppler_traces'))


## UTILITY

In [ ]:
import glob
import os
import numpy as np
import pickle
import math as mt
import shutil

def convert_to_number(lab, csi_label_dict):
    lab_num = np.argwhere(np.asarray(csi_label_dict) == lab)[0][0]
    return lab_num


def create_windows_antennas(csi_list, labels_list, sample_length, stride_length):
    # --- split each activity into windows (keep all antennas together)
    csi_matrix_stride = []
    labels_stride = []
    for i in range(len(labels_list)):        # iterate over the N ACTIVITIES
        csi_i = csi_list[i]                  # Doppler spectrum (all antennas) of activity i
        label_i = labels_list[i]             # its label (a single value)
        len_csi = csi_i.shape[2]             # time length = number of available Doppler columns
        # --- sliding window: start 0, step stride_length, stop when no full window fits
        for ii in range(0, len_csi - sample_length, stride_length):
            csi_wind = csi_i[:, :, ii:ii + sample_length, ...]   # crop [ii, ii+sample_length)
            csi_matrix_stride.append(csi_wind)   # +1 sample (one window)
            labels_stride.append(label_i)        # +1 label = the one of activity i
    return csi_matrix_stride, labels_stride

## GENERATE DATASET 

In [ ]:
# --- dataset generation (single train/test method)
#     complete=False -> temporal 60/20/20 split per activity ; complete=True -> whole sequence into "complete"

def generate_dataset(data_dir, list_subdir, num_packets, sliding, window_length,
                   stride_length, labels_activities, n_tot, complete=False):
    """
    data_dir         : root folder of the data (Doppler spectra)
    list_subdir      : experiment subfolders, comma-separated (e.g. "S1a,S1b,S1c")
    num_packets      : n. of Wi-Fi packets used for one STFT (e.g. 31)
    sliding          : step (in packets) between two consecutive STFTs
    window_length    : n. of Doppler columns per window (network input, e.g. 340)
    stride_length    : step between two consecutive windows
    labels_activities: activities considered, e.g. "E,J,L,R,W" (the index becomes the label)
    n_tot            : n. of spatial streams * n. of antennas (channels per activity, e.g. 4)
    complete         : False -> train/val/test split ; True -> single "complete" set
    """
    # --- label dictionary: E,J,L,R,W to list
    csi_label_dict = []
    for lab_act in labels_activities.split(','):
        csi_label_dict.append(lab_act)
    activities = np.asarray(labels_activities)   # only used to build output folder/file names

    # --- main loop: one subfolder at a time
    for subdir in list_subdir.split(','):
        exp_dir = data_dir + subdir + '/'  # full path of the experiment folder

        # --- output folder names
        path_train = exp_dir + 'train_antennas_' + str(activities)
        path_val = exp_dir + 'val_antennas_' + str(activities)
        path_test = exp_dir + 'test_antennas_' + str(activities)
        path_complete = exp_dir + 'complete_antennas_' + str(activities)

        # --- folder preparation (different logic in the two modes)
        if not complete:
            # train mode: prepare train/val/test (empty them if they exist, else create)
            for pat in [path_train, path_val, path_test]:
                if os.path.exists(pat):
                    for f in glob.glob(pat + '/*'):
                        os.remove(f)
                else:
                    os.mkdir(pat)
            # remove any "complete" from previous runs
            if os.path.exists(path_complete):
                shutil.rmtree(path_complete)
        else:
            # test mode: fully delete train/val/test and prepare "complete"
            for pat in [path_train, path_val, path_test]:
                if os.path.exists(pat):
                    shutil.rmtree(pat)
            if os.path.exists(path_complete):
                for f in glob.glob(path_complete + '/*'):
                    os.remove(f)
            else:
                os.mkdir(path_complete)

        # --- input file collection: files starting with 'S', without .txt, sorted (n_tot antennas end up consecutive)
        names = []
        all_files = os.listdir(exp_dir)
        for i in range(len(all_files)):
            if all_files[i].startswith('S'):
                names.append(all_files[i][:-4])
        names.sort()

        # --- grouping by activity: each activity = n_tot consecutive files (one per antenna)
        csi_matrices = []   # list of arrays (n_tot, doppler_freq, time) -> one per activity
        labels = []         # numeric label of each activity
        lengths = []        # temporal duration (n. of Doppler columns) of each activity
        label = 'null'
        prev_label = label
        csi_matrix = []     # buffer: the antennas of the current activity
        processed = False
        for i_name, name in enumerate(names):
            # --- at every multiple of n_tot (not first) an antenna group is complete
            if i_name % n_tot == 0 and i_name != 0 and processed:
                ll = csi_matrix[0].shape[1]
                for i_ant in range(1, n_tot):
                    if ll != csi_matrix[i_ant].shape[1]:
                        break
                lengths.append(ll)
                csi_matrices.append(np.asarray(csi_matrix))  # stack -> (n_tot, freq, time)
                labels.append(label)
                csi_matrix = []

            label = name[4]  # the label = 5th character of the file name
            if label not in csi_label_dict:
                processed = False
                continue
            processed = True
            print(name)

            label = convert_to_number(label, csi_label_dict)
            if i_name % n_tot == 0:
                prev_label = label
            elif label != prev_label:
                print('error in ' + str(name))
                break

            # --- load single-antenna Doppler spectrum, remove per-column mean
            name_file = exp_dir + name + '.txt'
            with open(name_file, "rb") as fp:
                stft_sum_1 = pickle.load(fp)
            stft_sum_1_mean = stft_sum_1 - np.mean(stft_sum_1, axis=0, keepdims=True)
            csi_matrix.append(stft_sum_1_mean.T)  # -> (doppler_freq, time)

        # --- close the LAST group (common to both modes)
        error = False
        if processed:
            if len(csi_matrix) < n_tot:
                print('error in ' + str(name))
            ll = csi_matrix[0].shape[1]
            for i_ant in range(1, n_tot):
                if ll != csi_matrix[i_ant].shape[1]:
                    print('error in ' + str(name))
                    error = True
            if not error:
                lengths.append(ll)
                csi_matrices.append(np.asarray(csi_matrix))
                labels.append(label)

        if error:
            continue

        # --- dispatch: split (train) or whole sequence (complete)
        if not complete:
            # train mode: temporal 60/20/20 split per activity
            lengths = np.asarray(lengths)
            csi_train, csi_val, csi_test = [], [], []
            length_train, length_val, length_test = [], [], []
            for i in range(len(labels)):
                ll = lengths[i]
                # train: first 60%
                train_len = int(np.floor(ll * 0.6))
                length_train.append(train_len)
                csi_train.append(csi_matrices[i][:, :, :train_len])
                # val: next 20%, with a ceil(num_packets/sliding) gap to avoid overlap
                start_val = train_len + mt.ceil(num_packets / sliding)
                val_len = int(np.floor(ll * 0.2))
                length_val.append(val_len)
                csi_val.append(csi_matrices[i][:, :, start_val:start_val + val_len])
                # test: remainder, after another gap
                start_test = start_val + val_len + mt.ceil(num_packets / sliding)
                length_test.append(ll - val_len - train_len - 2 * mt.ceil(num_packets / sliding))
                csi_test.append(csi_matrices[i][:, :, start_test:])

            list_sets_name = ['train', 'val', 'test']
            list_sets = [csi_train, csi_val, csi_test]
            list_sets_lengths = [length_train, length_val, length_test]

            # --- windowing + saving for each of the 3 sets
            for set_idx in range(3):
                csi_matrices_set, labels_set = create_windows_antennas(
                    list_sets[set_idx], labels, window_length, stride_length)
                num_windows = np.floor(
                    (np.asarray(list_sets_lengths[set_idx]) - window_length) / stride_length + 1)
                if not len(csi_matrices_set) == np.sum(num_windows):
                    print('ERROR - shapes mismatch')
                _save_windows(exp_dir, list_sets_name[set_idx], str(activities),
                                csi_matrices_set, labels_set, num_windows)
        else:
            # test mode: no split, the whole sequence into "complete"
            csi_complete = [csi_matrices[i] for i in range(len(labels))]
            csi_matrices_wind, labels_wind = create_windows_antennas(
                csi_complete, labels, window_length, stride_length)
            num_windows = np.floor((np.asarray(lengths) - window_length) / stride_length + 1)
            if not len(csi_matrices_wind) == np.sum(num_windows):
                print('ERROR - shapes mismatch')
            _save_windows(exp_dir, 'complete', str(activities),
                            csi_matrices_wind, labels_wind, num_windows)


def _save_windows(exp_dir, set_name, activities_str, csi_windows, labels_windows, num_windows):
    """Save the windows of a set + the summary files (labels_/files_/num_windows_)."""
    suffix = '.txt'
    names_set = []
    for ii in range(len(csi_windows)):
        name_file = exp_dir + set_name + '_antennas_' + activities_str + '/' + str(ii) + suffix
        names_set.append(name_file)
        with open(name_file, "wb") as fp:            # one window per file (pickle)
            pickle.dump(csi_windows[ii], fp)
    # labels_*: label of each window
    with open(exp_dir + '/labels_' + set_name + '_antennas_' + activities_str + suffix, "wb") as fp:
        pickle.dump(labels_windows, fp)
    # files_*: paths of the saved windows (used by the loader)
    with open(exp_dir + '/files_' + set_name + '_antennas_' + activities_str + suffix, "wb") as fp:
        pickle.dump(names_set, fp)
    # num_windows_*: n. of windows per activity (to reconstruct the antenna groups)
    with open(exp_dir + '/num_windows_' + set_name + '_antennas_' + activities_str + suffix, "wb") as fp:
        pickle.dump(num_windows, fp)


# --- parameters + call (non-complete mode: train/val/test split)
data_dir = '/content/doppler_traces/'   # root folder of the data (dataset unpacked on Colab)
list_subdir = 'S1a,S1b,S1c,S2a,S2b,S3a,S4a,S4b,S5a,S6a,S6b,S7a'   # number = SHARP set, letter = capture of same person/env
num_packets = 31                        # n. of Wi-Fi packets per STFT
sliding = 1                             # step (in packets) between two STFTs
window_length = 340                     # n. of Doppler columns per window (network input)
stride_length = 30                       # window stride: 30 for fine-tune/eval (more windows)
labels_activities = 'E,J,L,R,W'         # activities (the index = label: E->0, J->1, ...)
n_tot = 4                               # n. of streams * n. of antennas

# --- check: generate only if the dataset is not already present (FORCE_REGEN=True to force)
FORCE_REGEN = False


def dataset_exists(data_dir, list_subdir, activities_str, sets=('train', 'val', 'test')):
    """True ONLY if for every subfolder both the index AND the referenced windows exist.
       This way a 'stale' index (from an interrupted generation) forces regeneration."""
    for sdir in list_subdir.split(','):
        base = data_dir + sdir + '/'
        for st in sets:
            idx = base + 'files_' + st + '_antennas_' + activities_str + '.txt'
            if not os.path.isfile(idx):
                print(f'  missing index: {idx}')
                return False
            try:
                with open(idx, 'rb') as fp:
                    flist = pickle.load(fp)
            except Exception:
                print(f'  unreadable index: {idx}')
                return False
            # consistency check: first and last window must actually exist
            if len(flist) == 0 or not os.path.isfile(flist[0]) or not os.path.isfile(flist[-1]):
                print(f'  missing/inconsistent windows for {sdir} [{st}] -> regenerating')
                return False
    return True


activities_str = str(np.asarray(labels_activities))   # same format used in file names
if FORCE_REGEN or not dataset_exists(data_dir, list_subdir, activities_str):
    print('Generating the dataset...')
    generate_dataset(data_dir, list_subdir, num_packets, sliding, window_length,
                     stride_length, labels_activities, n_tot, complete=False)
    print('Dataset generated.')
else:
    print('Dataset already present: generation skipped (FORCE_REGEN=True to regenerate).')


## Data augmentation for contrastive learning
 

In [ ]:
import torch
import random
from torchvision import transforms


class ContrastiveTransformations(object):
    # --- apply base_transforms n_views times -> list of augmented views of the same window
    def __init__(self, base_transforms, n_views=1):
        self.base_transforms = base_transforms
        self.n_views = n_views

    def __call__(self, x):
        return [self.base_transforms(x) for i in range(self.n_views)]


# --- augmentation per single antenna (Doppler spectrogram); input (1, 340, 100) = (channel, TIME, DOPPLER)

class PerSampleStandardize(object):
    # --- mean 0 / std 1; clean NaN/inf and handle near-constant windows (std~0)
    def __call__(self, x):
        x = torch.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)
        std = x.std()
        if std < 1e-6:
            return x - x.mean()
        return (x - x.mean()) / std


class RandomTimeShift(object):
    # --- circular roll on the TIME axis (dim=-2); activity quasi-periodic -> valid shift
    def __init__(self, max_shift=30, p=0.5):
        self.max_shift = max_shift; self.p = p
    def __call__(self, x):
        if random.random() > self.p:
            return x
        s = random.randint(-self.max_shift, self.max_shift)
        return torch.roll(x, shifts=s, dims=-2)


class RandomTimeMask(object):
    # --- SpecAugment: zero a contiguous block on the TIME axis (dim=-2)
    def __init__(self, max_width=40, p=0.5):
        self.max_width = max_width; self.p = p
    def __call__(self, x):
        if random.random() > self.p:
            return x
        T = x.shape[-2]
        w = random.randint(1, self.max_width)
        t0 = random.randint(0, max(0, T - w))
        x = x.clone(); x[..., t0:t0 + w, :] = 0.0   # 0 = mean after standardize
        return x


class RandomFreqMask(object):
    # --- light SpecAugment on the DOPPLER axis (dim=-1); keep small (Doppler = velocity = core of activity and gait)
    def __init__(self, max_width=8, p=0.3):
        self.max_width = max_width; self.p = p
    def __call__(self, x):
        if random.random() > self.p:
            return x
        D = x.shape[-1]
        w = random.randint(1, self.max_width)
        d0 = random.randint(0, max(0, D - w))
        x = x.clone(); x[..., :, d0:d0 + w] = 0.0
        return x


class RandomGain(object):
    # --- random multiplicative amplitude scaling
    def __init__(self, lo=0.8, hi=1.2, p=0.5):
        self.lo = lo; self.hi = hi; self.p = p
    def __call__(self, x):
        if random.random() > self.p:
            return x
        return x * random.uniform(self.lo, self.hi)


class GaussianNoise(object):
    # --- additive Gaussian noise (sigma in standardized scale)
    def __init__(self, sigma=0.05, p=0.5):
        self.sigma = sigma; self.p = p
    def __call__(self, x):
        if random.random() > self.p:
            return x
        return x + torch.randn_like(x) * self.sigma


# --- pipeline: standardize first (known scale for sigma/gain), then augmentation
contrast_transforms = transforms.Compose([
    PerSampleStandardize(),                                   # mean 0 / std 1
    RandomTimeShift(max_shift=30, p=0.5),                     # temporal shift
    RandomTimeMask(max_width=40, p=0.5),                      # temporal mask
    RandomFreqMask(max_width=8, p=0.3),                       # Doppler mask (light)
    RandomGain(lo=0.8, hi=1.2, p=0.5),                        # amplitude scaling
    GaussianNoise(sigma=0.05, p=0.5),                         # additive noise
    transforms.RandomErasing(p=0.3, scale=(0.02, 0.1),       # cutout small patches
                             ratio=(0.3, 3.3), value=0.0),
])

## Multi-view dataset (4 antennas) + NT-Xent
Each window generates **4 views** (one per antenna), augmented independently. In multi-view NT-Xent, the **positives** of each view are the other 3 views of the **same window**; the **negatives** are all the views of the other windows in the batch.

In [ ]:
# --- contrastive multi-view dataset (all 4 antennas as views); each window -> n_views augmentations, one per antenna, independent
import pickle
import math as mt
import random
import numpy as np
from torch.utils.data import Dataset, DataLoader


class CSIContrastiveMultiView(Dataset):
    def __init__(self, csi_files, transform, n_views=4):
        self.files = csi_files
        self.transform = transform     # contrast_transforms (per single antenna)
        self.n_views = n_views         # <= n_tot ; default 4 = all antennas

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        with open(self.files[idx], "rb") as fp:
            matrix_csi = pickle.load(fp)          # (n_tot, 100, 340)
        n_ant = matrix_csi.shape[0]
        # --- pick n_views distinct antennas (n_views=4 -> all, random order)
        ants = random.sample(range(n_ant), self.n_views)
        views = []
        for a in ants:
            m = matrix_csi[a].T                    # (340, 100)  time x doppler
            m = np.ascontiguousarray(m[np.newaxis, ...])   # (1, 340, 100)
            x = torch.from_numpy(m).float()
            views.append(self.transform(x))        # independent augmentation per view, then i stack the antennas
        views = torch.stack(views, dim=0)          # (n_views, 1, 340, 100)
        return views, idx


# --- split config (set = numeric part of the subfolder name)
def set_num(subdir):
    # 'S6a' -> 6 ; 'S12b' -> 12
    return int(''.join(c for c in subdir[1:] if c.isdigit()))

subdirs_all = list_subdir.split(',')
# --- set held out only for the AR-generalization test
heldout_ar_set = 6
subdirs_pretrain = [s for s in subdirs_all if set_num(s) != heldout_ar_set]
print('Pretraining subfolders:', subdirs_pretrain)

# --- load TRAIN window paths + non-overlapping subsampling (1 window every k, k = ceil(window_length/stride_length))
csi_act = labels_activities                           # 'E,J,L,R,W'
suffix = '.txt'
labels_considered = np.arange(len(csi_act.split(',')))  # considered activities [0..n-1]
k_sub = mt.ceil(window_length / stride_length)         # 340/30 -> 12

contrastive_files = []
for sdir in subdirs_pretrain:
    base = data_dir + sdir + '/'
    with open(base + 'files_train_antennas_' + str(csi_act) + suffix, "rb") as fp:
        files_s = pickle.load(fp)
    with open(base + 'labels_train_antennas_' + str(csi_act) + suffix, "rb") as fp:
        labels_s = pickle.load(fp)
    # --- activity filter + non-overlap subsampling (every k-th window)
    files_filt = [f for f, l in zip(files_s, labels_s) if l in labels_considered]
    contrastive_files.extend(files_filt[::k_sub])


def make_contrastive_loader(csi_files, transform, batch_size, n_views=4, shuffle=True):
    ds = CSIContrastiveMultiView(csi_files, transform, n_views=n_views)
    # drop_last=True: NT-Xent wants full batches (constant number of negatives)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle,
                      num_workers=2, drop_last=True)


# --- safety filter: drop windows referenced but not present on disk
_before = len(contrastive_files)
contrastive_files = [f for f in contrastive_files if os.path.isfile(f)]
if len(contrastive_files) < _before:
    print(f'WARNING: {_before - len(contrastive_files)} missing windows filtered out '
          f'(regenerate with FORCE_REGEN=True if many).')

contrastive_loader = make_contrastive_loader(contrastive_files, contrast_transforms,
                                             batch_size=64, n_views=n_tot, shuffle=True)
print('Windows for pretraining (non-overlap):', len(contrastive_files))

In [ ]:
import torch
import torch.nn.functional as F

def nt_xent_multiview(z, n_views, temperature=0.2):
    """
    z           : (M, D) = (N * n_views, D) projection-head embeddings.
    n_views     : number of views per window (e.g. 4 antennas).
    temperature : temperature tau.

    Returns the scalar loss averaged over all M anchors.
    """
    M = z.size(0)                       # total rows = N * n_views
    N = M // n_views                    # number of distinct windows in the batch
    device = z.device

    # L2-normalize so the dot product is exactly the cosine similarity;
    z = F.normalize(z, dim=1)
    sim = (z @ z.t()) / temperature     # (M, M) pairwise cosine-sim / temperature

    # --- Build positive / self masks ---------------------------------------
    # group[i] = id of the window row i belongs to, e.g. [0,0,0,0, 1,1,1,1, ...]
    group = torch.arange(N, device=device).repeat_interleave(n_views)   # (M,)
    self_mask = torch.eye(M, dtype=torch.bool, device=device)           # diagonal (i == i)
    # positives = same window, excluding the anchor itself.
    pos_mask = (group[:, None] == group[None, :]) & ~self_mask          # (M, M) bool

    # --- denominator ------------------------------------------------
    # Drop the self-similarity term (set to -inf so exp(-inf) = 0), then the
    # log-softmax over the row gives log P(j | i) with the standard InfoNCE
    # normalization (sum over every column except i).
    sim = sim.masked_fill(self_mask, float('-inf'))
    log_prob = sim - torch.logsumexp(sim, dim=1, keepdim=True)          # (M, M)

    # --- Average log-prob over the positive set (SupCon multi-positive) -----
    pos_per_row = pos_mask.sum(1).clamp(min=1)                          # |P(i)|, avoid /0
    # Keep log_prob only on positive entries, 0 elsewhere. Using torch.where
    # (instead of log_prob * pos_mask) avoids -inf * 0 = NaN on masked cells.
    log_prob_pos = torch.where(pos_mask, log_prob, torch.zeros_like(log_prob))
    loss = -log_prob_pos.sum(1) / pos_per_row                          # per-anchor loss L_i
    return loss.mean()                                                 # batch mean


# CONTRASTIVE PRETRAINING + 2 HEADS (AR + PI)
Pipeline: **self-supervised pretraining** (multi-view NT-Xent, 4 antennas) of a shared backbone, then **2-head fine-tuning** (AR = activity, PI = person).

In [ ]:

import torchvision
import torch
import torch.nn as nn
import torch.nn.functional as F

# --- independent setup: parameters used by the contrastive section (no need to re-run the supervised training cell)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
sample_length = window_length                       # 340 (time axis)
feature_length = 100                                # 100 (Doppler axis)
output_shape = len(labels_activities.split(','))    # n. activities (E,J,L,R,W -> 5)

In [ ]:
EMBED_DIM = 256    # embedding dimension shared by the 2 backbones


class ResNetEncoder(nn.Module):

    def __init__(self, input_channels=1, embed_dim=EMBED_DIM):
        super().__init__()
        net = torchvision.models.resnet34(weights=None)
        # --- patch conv1: from 3 channels (RGB) to 1 channel (Doppler spectrogram)
        net.conv1 = nn.Conv2d(input_channels, 64, kernel_size=7, stride=2, padding=3, bias=False)
        # --- fc: from classification head to projection 512 -> embed_dim
        net.fc = nn.Linear(net.fc.in_features, embed_dim)
        self.net = net
        self.feature_dim = embed_dim

    def forward(self, x):
        return self.net(x)                                                   # (N, embed_dim)


# --- shape sanity check (instantiate each once)
_res = ResNetEncoder()
_dummy = torch.zeros(2, 1, sample_length, feature_length)
print('ResNetEncoder    feature_dim:', _res.feature_dim, '| out:', tuple(_res(_dummy).shape))

In [ ]:
# --- projection head + ContrastiveNet (SimCLR); head used only in pretraining, then keep the backbone only
class ProjectionHead(nn.Module):
    def __init__(self, in_dim, hidden_dim=512, proj_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, proj_dim),
        )

    def forward(self, x):
        return self.net(x)


class ContrastiveNet(nn.Module):
    def __init__(self, encoder, hidden_dim=512, proj_dim=128):
        super().__init__()
        self.encoder = encoder
        self.projection = ProjectionHead(encoder.feature_dim, hidden_dim, proj_dim)

    def forward(self, x):
        return self.projection(self.encoder(x))       # (N, proj_dim)

In [ ]:
# --- contrastive pretraining loop
def pretrain_contrastive(encoder, loader, epochs=100, lr=2e-4, weight_decay=1e-4,
                         temperature=0.15, hidden_dim=512, proj_dim=128,
                         ckpt_name='contrastive.torch', device=device):
    torch.cuda.empty_cache()
    model = ContrastiveNet(encoder, hidden_dim, proj_dim).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=lr / 50)
    use_amp = False   # AMP off: Inception without BatchNorm -> fp16 can overflow (NaN)
    scaler = torch.amp.GradScaler('cuda', enabled=use_amp)   # new API (not deprecated)

    start_epoch = 0
    if os.path.exists(ckpt_name):                       # RESUME (Colab interrupts sessions)
        ck = torch.load(ckpt_name, map_location=device)
        model.load_state_dict(ck['model_state_dict'])
        optimizer.load_state_dict(ck['optimizer_state_dict'])
        scheduler.load_state_dict(ck['scheduler_state_dict'])
        if 'scaler_state_dict' in ck:
            scaler.load_state_dict(ck['scaler_state_dict'])
        start_epoch = ck['epoch'] + 1
        print(f'Resume {ckpt_name} from epoch {start_epoch}')

    for epoch in range(start_epoch, epochs):
        model.train()
        losses, n_bad = [], 0
        for views, _ in loader:                        # views: (N, V, 1, 340, 100)
            N, V = views.shape[0], views.shape[1]
            # view(N*V, ...): the V views of a window stay CONSECUTIVE (required by nt_xent_multiview)
            x = views.view(N * V, 1, sample_length, feature_length).to(device)
            optimizer.zero_grad()
            with torch.amp.autocast('cuda', enabled=use_amp):
                z = model(x)                           # (N*V, proj_dim)
            loss = nt_xent_multiview(z.float(), n_views=V, temperature=temperature)  # loss in fp32 (stable)
            if not torch.isfinite(loss):    # guard: do not corrupt weights with NaN/inf
                n_bad += 1
                continue
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            losses.append(loss.item())
        scheduler.step()
        mean_loss = float(np.mean(losses)) if losses else float('nan')
        msg = f'[{ckpt_name}] Epoch {epoch + 1}/{epochs} - contrastive_loss={mean_loss:.4f}'
        if n_bad:
            msg += f' (non-finite batches dropped: {n_bad})'
        print(msg)
        torch.save({'epoch': epoch,
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'scheduler_state_dict': scheduler.state_dict(),
                    'scaler_state_dict': scaler.state_dict()}, ckpt_name)
    return model.encoder

In [ ]:
# --- checkpoint dir + pre-trained weights
#     USE_OWN_DRIVE=True also makes the checkpoints saved during training survive a Colab drop.
import os, glob, shutil

USE_OWN_DRIVE = False   # True -> mount your Drive and read/save the checkpoints there

# name -> exact size in bytes (a truncated checkpoint would silently restart training from 0)
WEIGHTS = {
    'contrastive_resnet.torch':  259553976,
    'twohead_resnet.torch':      258794795,
    'twohead_resnet_best.torch':  86334911,
}

# --- destination dir (must be writable: training saves here every epoch)
if USE_OWN_DRIVE:
    from google.colab import drive
    if not os.path.ismount('/content/drive'):
        drive.mount('/content/drive')
    CKPT_DIR = '/content/drive/MyDrive/HAR_checkpoints'
else:
    CKPT_DIR = '/content/HAR_checkpoints'
os.makedirs(CKPT_DIR, exist_ok=True)


def missing():
    return [n for n, s in WEIGHTS.items() if not ok_size(os.path.join(CKPT_DIR, n), s)]


def adopt(name, src):
    """Copy src into CKPT_DIR only if the size matches; never overwrite a valid local file."""
    dst = os.path.join(CKPT_DIR, name)
    if ok_size(dst, WEIGHTS[name]) or not ok_size(src, WEIGHTS[name]):
        return False
    if os.path.abspath(src) == os.path.abspath(dst):
        return False
    shutil.copy2(src, dst)
    print(f'  {name}: taken from {src}')
    return True

# ----------------------------------------- TO NOT UPLOAD PRE-TRAINED WEIGHTS, COMMENT THIS SECTION -----------------------------------------

# --- source 1: mounted Drive (your own copy, or a shortcut to a shared folder)
if missing() and os.path.ismount('/content/drive'):
    for name in list(missing()):
        # bounded search: My Drive root + immediate subfolders
        # (a fully recursive glob over the FUSE mount is extremely slow)
        for cand in (glob.glob(f'/content/drive/MyDrive/{name}')
                     + glob.glob(f'/content/drive/MyDrive/*/{name}')):
            if adopt(name, cand):
                break

# --- source 2: local copy left by a previous run in this session
if missing():
    for name in list(missing()):
        adopt(name, os.path.join('/content/HAR_checkpoints', name))

# --- source 3: GitHub Release (fetch() and ok_size() come from the first cell)
for name in list(missing()):
    fetch(name, WEIGHTS[name], out_dir=CKPT_DIR)

print('Checkpoint dir:', CKPT_DIR)
for n in sorted(os.listdir(CKPT_DIR)):
    print(f'  {n:28s} {os.path.getsize(os.path.join(CKPT_DIR, n)) / 1e6:8.1f} MB')
assert not missing(), f'incomplete weights: {missing()} (re-run this cell: the download resumes)'

# -------------------------------------------------------------------------------------------------------------------------------------------

def ckpt_path(name):
    return os.path.join(CKPT_DIR, name)


In [ ]:
# --- pretraining driver: ResNet34 backbone
PRETRAIN_EPOCHS = 100
CONTRASTIVE_TEMP = 0.15

# --- backbone: ResNet34
res_encoder = ResNetEncoder(input_channels=1)
res_backbone = pretrain_contrastive(res_encoder, contrastive_loader,
                                    epochs=PRETRAIN_EPOCHS, temperature=CONTRASTIVE_TEMP,
                                    ckpt_name=ckpt_path('contrastive_resnet.torch'))

In [ ]:
# --- person label (PI) + 2-head dataset; person label derived from the set, single-antenna input
import re

# --- set -> person map (contiguous integer ids);
set_to_person = {1: 0, 2: 0, 3: 1, 4: 0, 5: 1, 6: 0, 7: 2} 

n_act = output_shape                                   # n. activities (5)
n_persons = len(set(set_to_person[set_num(s)] for s in subdirs_pretrain))
print('n_act:', n_act, '| n_persons (in the training sets):', n_persons)


class CSITwoHeadDataset(Dataset):
    """Single-antenna: returns (x, y_activity, y_person)."""
    def __init__(self, files, act_labels, person_labels, stream_ant):
        self.files = files
        self.act = act_labels
        self.person = person_labels
        self.stream_ant = list(stream_ant)

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        with open(self.files[idx], "rb") as fp:
            m = pickle.load(fp)                        # (4, 100, 340)
        s = self.stream_ant[idx]
        x = m[s].T                                     # (340, 100)
        x = torch.from_numpy(np.ascontiguousarray(x[np.newaxis, ...])).float()  # (1,340,100)
        y_a = torch.tensor(int(self.act[idx]), dtype=torch.long)
        y_p = torch.tensor(int(self.person[idx]), dtype=torch.long)
        return x, y_a, y_p


def load_split_twohead(subdirs, set_name):
    """Load a set (train/val/test) over multiple subfolders, with activity + person labels,
       and expand per antenna (single-antenna input, as in the main)."""
    files_sel, act_sel, per_sel = [], [], []
    for sdir in subdirs:
        base = data_dir + sdir + '/'
        with open(base + 'files_' + set_name + '_antennas_' + str(csi_act) + suffix, "rb") as fp:
            fs = pickle.load(fp)
        with open(base + 'labels_' + set_name + '_antennas_' + str(csi_act) + suffix, "rb") as fp:
            ls = pickle.load(fp)
        p = set_to_person[set_num(sdir)]              # person of the set
        for f, l in zip(fs, ls):
            if l in labels_considered and os.path.isfile(f):
                files_sel.append(f); act_sel.append(l); per_sel.append(p)
    # --- per-antenna expansion: each window -> n_tot single-antenna samples
    files_exp = [f for f in files_sel for _ in range(n_tot)]
    act_exp   = [a for a in act_sel   for _ in range(n_tot)]
    per_exp   = [p for p in per_sel   for _ in range(n_tot)]
    stream_ant = np.tile(np.arange(n_tot), len(files_sel))
    return files_exp, act_exp, per_exp, stream_ant


def make_twohead_loader(subdirs, set_name, batch_size=32, shuffle=False):
    fx, ax, px, sa = load_split_twohead(subdirs, set_name)
    ds = CSITwoHeadDataset(fx, ax, px, sa)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, num_workers=2, drop_last=False)


# --- split for the 2 heads
subdirs_heldout = [s for s in subdirs_all if set_num(s) == heldout_ar_set]   # AR-gen only

ft_train_loader = make_twohead_loader(subdirs_pretrain, 'train', batch_size=32, shuffle=True)
ft_val_loader   = make_twohead_loader(subdirs_pretrain, 'val',   batch_size=32, shuffle=False)
# PI + within-domain AR test (seen persons, windows held-out in time):
test_within_loader = make_twohead_loader(subdirs_pretrain, 'test', batch_size=32, shuffle=False)
# AR generalization test (held-out set, new environment/day): union of train+val+test of the held-out set
ar_gen_loaders = [make_twohead_loader(subdirs_heldout, s, batch_size=32, shuffle=False)
                  for s in ['train', 'val', 'test']]
print('Held-out set for AR-gen:', subdirs_heldout)

In [ ]:
# --- TwoHeadNet: shared backbone + 2 MLP heads (AR and PI)
def make_mlp_head(in_dim, out_dim, hidden_dim=None, p_drop=0.0):
    # --- MLP head: Linear -> BN -> ReLU -> (Dropout) -> Linear
    hidden_dim = hidden_dim or in_dim                     # default: same feature dim
    layers = [
        nn.Linear(in_dim, hidden_dim),
        nn.BatchNorm1d(hidden_dim),
        nn.ReLU(inplace=True),
    ]
    if p_drop > 0:
        layers.append(nn.Dropout(p_drop))                 # regularizes fine-tune
    layers.append(nn.Linear(hidden_dim, out_dim))
    return nn.Sequential(*layers)

class TwoHeadNet(nn.Module):
    def __init__(self, encoder, n_act, n_persons, hidden_dim=None, p_drop=0.0):
        super().__init__()
        self.encoder = encoder
        self.head_ar = make_mlp_head(encoder.feature_dim, n_act,     hidden_dim, p_drop)   # activity
        self.head_pi = make_mlp_head(encoder.feature_dim, n_persons, hidden_dim, p_drop)   # person

    def forward(self, x):
        f = self.encoder(x)
        return self.head_ar(f), self.head_pi(f)                    # (logits_ar, logits_pi)

In [ ]:
# --- 2-head fine-tuning: phase A linear probe (frozen backbone) + phase B full fine-tune; loss = CE(AR) + lam*CE(PI); resumable checkpoint + early stopping
def _run_epoch(model, loader, optimizer, ce, lam, train, device, encoder_eval=False):
    if train:
        model.train()
        if encoder_eval:
            model.encoder.eval()          # probe: also freeze the encoder's BatchNorm
    else:
        model.eval()
    tot, loss_sum, ca, cp = 0, 0.0, 0, 0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for x, ya, yp in loader:
            x, ya, yp = x.to(device), ya.to(device), yp.to(device)
            la, lp = model(x)
            loss = ce(la, ya) + lam * ce(lp, yp)
            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            bs = x.size(0); tot += bs; loss_sum += loss.item() * bs
            ca += (la.argmax(1) == ya).sum().item()
            cp += (lp.argmax(1) == yp).sum().item()
    return loss_sum / tot, ca / tot, cp / tot


def finetune_twohead(encoder, train_loader, val_loader, n_act, n_persons, lam=1.0,
                     probe_epochs=15, full_epochs=10, lr_probe=1e-3, lr_full=1e-4,
                     wd=1e-4, patience=5, ckpt_name='twohead.torch', device=device):
    model = TwoHeadNet(encoder, n_act, n_persons).to(device)
    ce = nn.CrossEntropyLoss()
    best_path = ckpt_name.replace('.torch', '_best.torch')   # best weights only (for the final load)

    # --- resumable state
    phase, start_epoch, best, no_improve, ck = 'probe', 0, -1.0, 0, None
    if os.path.exists(ckpt_name):                            # RESUME after Colab disconnection
        ck = torch.load(ckpt_name, map_location=device)
        model.load_state_dict(ck['model_state_dict'])
        phase, start_epoch, best = ck['phase'], ck['epoch'], ck['best']
        no_improve = ck.get('no_improve', 0)                 # consecutive epochs without improvement
        print(f'Resume {ckpt_name}: phase={phase} epoch={start_epoch} best={best:.3f} no_improve={no_improve}')

    def save_ck(cur_phase, next_epoch, opt):
        # --- full checkpoint: everything needed to resume training
        torch.save({'phase': cur_phase, 'epoch': next_epoch, 'best': best, 'no_improve': no_improve,
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': opt.state_dict(),
                    'n_act': n_act, 'n_persons': n_persons}, ckpt_name)

    # --- phase A: linear probe (frozen backbone)
    if phase == 'probe':
        for p in model.encoder.parameters():
            p.requires_grad = False
        opt = torch.optim.AdamW([*model.head_ar.parameters(), *model.head_pi.parameters()],
                                lr=lr_probe, weight_decay=1e-3)
        if ck is not None and ck['phase'] == 'probe':
            opt.load_state_dict(ck['optimizer_state_dict'])  # resume the probe optimizer state
        for e in range(start_epoch, probe_epochs):
            _run_epoch(model, train_loader, opt, ce, lam, True, device, encoder_eval=True)
            vl = _run_epoch(model, val_loader, opt, ce, lam, False, device)
            print(f'[probe {e + 1}/{probe_epochs}] val_loss={vl[0]:.3f} AR={vl[1]:.3f} PI={vl[2]:.3f}')
            save_ck('probe', e + 1, opt)
        phase, start_epoch, ck = 'full', 0, None             # move to phase B (new optimizer)

    # --- phase B: full fine-tune (unfreeze backbone, small LR)
    for p in model.encoder.parameters():
        p.requires_grad = True
    opt = torch.optim.AdamW(model.parameters(), lr=lr_full, weight_decay=wd)
    if ck is not None and ck['phase'] == 'full':
        opt.load_state_dict(ck['optimizer_state_dict'])      # resume the full optimizer state
    for e in range(start_epoch, full_epochs):
        _run_epoch(model, train_loader, opt, ce, lam, True, device)
        vl = _run_epoch(model, val_loader, opt, ce, lam, False, device)
        score = vl[1] + vl[2]                                # implicit AR+PI average
        if score > best:
            best = score
            no_improve = 0                                   # improved: reset counter
            torch.save(model.state_dict(), best_path)        # keeps the best weights on val
        else:
            no_improve += 1                                  # no improvement
        print(f'[full  {e + 1}/{full_epochs}] val_loss={vl[0]:.3f} AR={vl[1]:.3f} PI={vl[2]:.3f}'
              f' | best={best:.3f} no_improve={no_improve}/{patience}')
        save_ck('full', e + 1, opt)                          # resumable checkpoint every epoch
        if no_improve >= patience:                           # EARLY STOPPING
            print(f'Early stopping: no improvement on val for {patience} epochs.')
            break

    if os.path.exists(best_path):                            # load the best weights on val
        model.load_state_dict(torch.load(best_path, map_location=device))
    return model


# --- fine-tuning of the 2 backbones (uses the pre-trained backbones)

res_model = finetune_twohead(res_backbone, ft_train_loader, ft_val_loader, n_act, n_persons,
                             ckpt_name=ckpt_path('twohead_resnet.torch'))

In [ ]:
# --- evaluation + decision fusion (both heads AR and PI): mean softmax over the n_tot antennas of the same acquisition
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

def predict_two(model, loader, device=device):
    model.eval()
    Pa, Pp, Ya, Yp = [], [], [], []
    with torch.no_grad():
        for x, ya, yp in loader:
            la, lp = model(x.to(device))
            Pa.append(F.softmax(la, 1).cpu().numpy())
            Pp.append(F.softmax(lp, 1).cpu().numpy())
            Ya.append(ya.numpy()); Yp.append(yp.numpy())
    return (np.concatenate(Pa), np.concatenate(Pp),
            np.concatenate(Ya), np.concatenate(Yp))


def _fuse(prob, n=n_tot):
    # --- mean softmax over blocks of n antennas -> (N_acq, C)
    N = prob.shape[0] // n
    return prob[:N * n].reshape(N, n, -1).mean(1)


def eval_task(prob, y_true, n=n_tot):
    fused = _fuse(prob, n)
    N = fused.shape[0]
    yt = np.asarray(y_true)[:N * n].reshape(N, n)[:, 0]   # 1 label per acquisition
    yp = fused.argmax(1)
    return accuracy_score(yt, yp), f1_score(yt, yp, average='macro'), confusion_matrix(yt, yp)


def eval_pi_gen(prob, y_true, n_persons, n=n_tot):
    # --- PI on held-out set = 1 known person -> cross-domain robustness check (accuracy + distribution), not discrimination
    fused = _fuse(prob, n)
    N = fused.shape[0]
    yt = np.asarray(y_true)[:N * n].reshape(N, n)[:, 0]
    yp = fused.argmax(1)
    acc = accuracy_score(yt, yp)
    persons_true = sorted(set(yt.tolist()))
    pred_counts = {int(k): int((yp == k).sum()) for k in range(n_persons)}
    return acc, persons_true, pred_counts, N


def evaluate_model(name, model):
    # --- within-domain test (seen persons, windows held-out in time): AR + PI
    Pa, Pp, Ya, Yp = predict_two(model, test_within_loader)
    ar_acc, ar_f1, ar_cm = eval_task(Pa, Ya)
    pi_acc, pi_f1, pi_cm = eval_task(Pp, Yp)
    # --- generalization on held-out set (new environment/day): AR + PI
    Pa_g, Ya_g, Pp_g, Yp_g = [], [], [], []
    for ld in ar_gen_loaders:
        pa, pp, ya, yp = predict_two(model, ld)
        Pa_g.append(pa); Ya_g.append(ya); Pp_g.append(pp); Yp_g.append(yp)
    arg_acc, arg_f1, _ = eval_task(np.concatenate(Pa_g), np.concatenate(Ya_g))
    # --- PI-gen: weak test (held-out set = 1 known person), cross-domain robustness only
    pig_acc, pig_persons, pig_pred, pig_N = eval_pi_gen(
        np.concatenate(Pp_g), np.concatenate(Yp_g), n_persons)
    print(f'==== {name} ====')
    print(f'AR within        acc={ar_acc:.3f}  f1={ar_f1:.3f}')
    print(f'PI within        acc={pi_acc:.3f}  f1={pi_f1:.3f}')
    print(f'AR gen(hold-out) acc={arg_acc:.3f}  f1={arg_f1:.3f}')
    print(f'PI gen(hold-out) acc={pig_acc:.3f}  [WEAK TEST: persons present={pig_persons}, '
          f'N_acq={pig_N}]')
    print(f'   -> PI-gen prediction distribution per class: {pig_pred}')
    print('   -> acc = fraction still recognized as the correct person in a new environment;')
    print('      does NOT measure discrimination between persons (1 single class in the held-out set).')
    print('Confusion AR (within):\n', ar_cm)
    print('Confusion PI (within):\n', pi_cm)
    return dict(AR_within=ar_acc, PI_within=pi_acc, AR_gen=arg_acc, PI_gen=pig_acc,
                AR_f1=ar_f1, PI_f1=pi_f1, ARgen_f1=arg_f1)


res_res = evaluate_model('ResNet', res_model)

# --- ResNet metrics summary
print('\n=== ResNet METRICS ===')
print(pd.DataFrame([res_res], index=['ResNet']).round(3))
